# PATSTĀVĪGAIS DARBS
## 2. nedēļa: Regresija
*Nepārtrauktas vērtības prognozēšana, vizualizācija un kļūdu metrikas*

---
## 1. uzdevums — Sagatavo datus regresijai

### 1.1. Ielādē datu kopu

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# shoppers_clean.csv atrodas projekta saknes mapē (nevis week1/)
df = pd.read_csv('../shoppers_clean.csv')
print(f"Datu kopa: {df.shape[0]} rindas, {df.shape[1]} kolonnas")
df.head()

### 1.2. Izvēlies target un features — PageValues sadalījums

In [ ]:
print("PageValues statistika:")
print(df['PageValues'].describe())

plt.figure(figsize=(8, 4))
df['PageValues'].hist(bins=50, color='steelblue', edgecolor='white')
plt.title('PageValues sadalījums')
plt.xlabel('PageValues')
plt.ylabel('Biežums')
plt.tight_layout()
plt.savefig('pagevalues_distribution.png', dpi=120)
plt.show()

### 1.3. Sagatavo X un y

In [ ]:
y = df['PageValues']
X = df.drop(['PageValues', 'Revenue'], axis=1)

print(f"Features skaits: {X.shape[1]}")
print(f"Target: PageValues")
print(f"\nFeature nosaukumi:")
print(X.columns.tolist())

### Kāpēc izņēmām `Revenue` no features? — Data Leakage skaidrojums

**Data leakage** (datu noplūde) rodas, kad modelim trenēšanas laikā ir pieejama informācija, kurai reālā praksē nebūtu jābūt pieejamai prognozes brīdī.

`Revenue` kolonna satur informāciju par to, vai klients **faktiski veica pirkumu**. Šī informācija ir cieši saistīta ar `PageValues` — ja klients veica pirkumu, tad lapas vērtība parasti bija augsta. Ja mēs iekļautu `Revenue` kā feature, modelis faktiski "redzētu atbildi" jau ievaddatos un mācītos no nākotnes informācijas.

**Kāpēc tas ir bīstami?**
- Modelis uzrāda nereālistiski labus rezultātus treniņa un validācijas laikā
- Reālā vidē `Revenue` vērtība nav zināma pirms sesijas beigām — tātad praksē šo feature nevar izmantot
- Modelis ir nelietojams — tas ir iemācījies "krāpties", nevis reālas sakarības

**Secinājums:** `Revenue` ir jāizslēdz no features, lai modelis mācītos no informācijas, kas ir pieejama **pirms** sesijas beigām — tāpat kā tas strādātu reālā biznesā.

### 1.4. Treniņa/testa sadalījums

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Treniņa kopa: {X_train.shape[0]} rindas")
print(f"Testa kopa:   {X_test.shape[0]} rindas")

---
## 2. uzdevums — Uztrenē lineārās regresijas modeli

### 2.1. Modeļa trenēšana un metrikas

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

y_pred = lr_model.predict(X_test)

mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print("=== Lineāra regresija: rezultāti ===")
print(f"MSE:  {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")
print(f"R²:   {r2:.4f}")

### 2.2. Metriku interpretācija

*(Piezīme: `shoppers_clean.csv` PageValues ir StandardScaler normalizēts — tāpēc vienības ir standartnovirzes, nevis sākotnējās lapas vērtības.)*

**MAE = 0.5009** — vidējā absolūtā kļūda:
- Modelis vidēji kļūdās par ~0.50 standartnovirzēm no reālās PageValues vērtības.
- Visas kļūdas tiek ņemtas vienādi — gan mazas, gan lielas — tāpēc MAE ir viegli interpretējams.

**RMSE = 0.9946** — kļūdu kvadrātu vidējā sakne:
- Lielākas kļūdas tiek *sodītas vairāk*, jo kļūda tiek kvadrēta pirms vidējošanas. Kļūda 1.0 iegūst svaru 1.0, bet kļūda 3.0 — svaru 9.0.
- Tā kā RMSE (0.9946) ir ievērojami lielāks par MAE (0.5009), tas liecina, ka ir atsevišķi gadījumi ar lielām kļūdām (piemēram, sesijas ar ļoti augstu PageValues).

**R² = 0.0456** — determinācijas koeficients:
- Modelis izskaidro tikai ~4.6% no PageValues variācijas. R² = 1.0 būtu ideāls modelis, R² = 0.0 — nav labāks par vienkāršu vidējo.
- Šis ir ļoti zems R², kas nozīmē, ka lineāra regresija nespēj labi uztvert PageValues.

**Vai modelis ir labs vai slikts?**
- R² = 0.0456 ir ļoti vājš rezultāts — modelis gandrīz neizskaidro PageValues variāciju.
- Iemesls: `PageValues` (pēc StandardScaler) ir ļoti asimetrisks sadalījums — lielākā daļa vērtību ir pie minimuma (−0.317), ar retiem augstiem izņēmumiem. Lineāra regresija grūti tiek galā ar šādu sadalījumu.
- Modelis der kā bāzlīnija, bet praktiski nav lietojams prognozēšanai bez uzlabojumiem.

---
## 3. uzdevums — Vizualizē prognozes

### 3.1. Faktiskās vs. prognozētās vērtības

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.3, color='steelblue', edgecolors='white', s=30)

max_val = max(y_test.max(), y_pred.max())
plt.plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='Ideālā prognoze')

plt.xlabel('Faktiskā vērtība')
plt.ylabel('Prognozētā vērtība')
plt.title('Lineāra regresija: Faktiskās vs. Prognozētās vērtības')
plt.legend()
plt.tight_layout()
plt.savefig('lr_actual_vs_predicted.png', dpi=120)
plt.show()

### 3.2. Atlikumu (residuals) analīze

In [ ]:
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(residuals, bins=50, color='coral', edgecolor='white')
axes[0].set_title('Atlikumu sadalījums')
axes[0].set_xlabel('Atlikums (faktiskā − prognozētā)')
axes[0].set_ylabel('Biežums')
axes[0].axvline(x=0, color='black', linestyle='--')

axes[1].scatter(y_pred, residuals, alpha=0.3, color='coral', edgecolors='white', s=30)
axes[1].axhline(y=0, color='black', linestyle='--')
axes[1].set_title('Atlikumi vs. Prognozētās vērtības')
axes[1].set_xlabel('Prognozētā vērtība')
axes[1].set_ylabel('Atlikums')

plt.tight_layout()
plt.savefig('lr_residuals.png', dpi=120)
plt.show()

### 3.3. Vizualizāciju interpretācija

**Scatter grafiks (faktiskās vs. prognozētās vērtības):**
- Punkti nav tuvu sarkanai ideālās prognozes līnijai — lielākā daļa ir sagrupēta ap 0, bet augstākām faktiskajām vērtībām modelis prognozē pārāk zemu. Tas apstiprina zemo R² (4.6%) — modelis praktiski neprognozē augstās PageValues vērtības.

**Atlikumu histogramma:**
- Atlikumiem vajadzētu būt simetriskiem ap 0 (normāli sadalītiem). Faktiski histogramma ir starki nobīdīta pa kreisi — tas nozīmē, ka modelis sistemātiski *pārprognozē* (prognozē augstāk nekā realitāte) gadījumos ar zemu PageValues.
- Garā aste pa labi (lielās pozitīvās kļūdas) rodas tur, kur reālā vērtība ir augsta, bet modelis to nespēj prognozēt.

**Atlikumi vs. prognozētās vērtības:**
- Skaidri redzama piltuves forma — pie zemām prognozētajām vērtībām atlikumi ir sakoncentrēti, bet pie augstākām — izkaisīti. Tas liecina par **heteroskedasticitāti**: modelis ir ievērojami mazāk uzticams augsto PageValues reģionā.

**Kopsecinājums:** Vizualizācijas skaidri parāda, ka lineārā regresija nespēj uztvert PageValues sadalījuma nelineāro raksturu. Modelis ir konservatīvs — tas cenšas prognozēt ap vidējo vērtību, bet klients ar augstu PageValues tam sagādā lielas kļūdas.

---
## 4. uzdevums — Izmēģini polinomiālo regresiju

### 4.1. PolynomialFeatures ar degree=2

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline

poly_pipeline = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('lr', LinearRegression())
])

poly_pipeline.fit(X_train, y_train)
y_pred_poly = poly_pipeline.predict(X_test)

mae_poly  = mean_absolute_error(y_test, y_pred_poly)
rmse_poly = np.sqrt(mean_squared_error(y_test, y_pred_poly))
r2_poly   = r2_score(y_test, y_pred_poly)

print("=== Polinomiālā regresija (degree=2): rezultāti ===")
print(f"MAE:  {mae_poly:.4f}")
print(f"RMSE: {rmse_poly:.4f}")
print(f"R²:   {r2_poly:.4f}")

### 4.2. Salīdzinājuma tabula

In [ ]:
comparison = pd.DataFrame({
    'Modelis': ['Lineārā regresija', 'Polinomiālā regresija (degree=2)'],
    'MAE':  [mae,  mae_poly],
    'RMSE': [rmse, rmse_poly],
    'R²':   [r2,   r2_poly]
}).set_index('Modelis').round(4)

print(comparison.to_string())
comparison

### 4.3. Overfitting pārbaude — Train vs. Test R²

In [ ]:
r2_lr_train   = lr_model.score(X_train, y_train)
r2_lr_test    = lr_model.score(X_test,  y_test)
r2_poly_train = poly_pipeline.score(X_train, y_train)
r2_poly_test  = poly_pipeline.score(X_test,  y_test)

print("=== Overfitting pārbaude ===")
print(f"Lineārā regresija    — Train R²: {r2_lr_train:.4f} | Test R²: {r2_lr_test:.4f} | Starpība: {abs(r2_lr_train - r2_lr_test):.4f}")
print(f"Polinomiālā regresija — Train R²: {r2_poly_train:.4f} | Test R²: {r2_poly_test:.4f} | Starpība: {abs(r2_poly_train - r2_poly_test):.4f}")

threshold = 0.10
diff_lr   = abs(r2_lr_train   - r2_lr_test)
diff_poly = abs(r2_poly_train - r2_poly_test)

if diff_lr > threshold:
    print(f"\n⚠️  BRĪDINĀJUMS: Lineārā — starpība ({diff_lr:.4f}) > {threshold} → iespējams overfitting!")
else:
    print(f"\n✅  Lineārā: starpība ({diff_lr:.4f}) ir pieņemama.")

if diff_poly > threshold:
    print(f"⚠️  BRĪDINĀJUMS: Polinomiālā — starpība ({diff_poly:.4f}) > {threshold} → iespējams overfitting!")
else:
    print(f"✅  Polinomiālā: starpība ({diff_poly:.4f}) ir pieņemama.")

### 4.4. Vizuāls salīdzinājums

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

models_info = [
    ('Lineārā regresija',             y_pred,      'steelblue'),
    ('Polinomiālā regresija (deg=2)',  y_pred_poly, 'mediumpurple'),
]

for ax, (title, preds, color) in zip(axes, models_info):
    ax.scatter(y_test, preds, alpha=0.3, color=color, edgecolors='white', s=30)
    max_val = max(y_test.max(), float(np.array(preds).max()))
    ax.plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='Ideālā prognoze')
    ax.set_title(title)
    ax.set_xlabel('Faktiskā vērtība')
    ax.set_ylabel('Prognozētā vērtība')
    ax.legend()

plt.suptitle('Lineārā vs. Polinomiālā regresija: Faktiskās vs. Prognozētās vērtības', fontsize=13)
plt.tight_layout()
plt.savefig('comparison_actual_vs_predicted.png', dpi=120, bbox_inches='tight')
plt.show()

### 4.5. Secinājumi

**Kurš modelis labāk prognozē PageValues?**

Salīdzinot metriku tabulu: lineārā regresija (MAE=0.5009, RMSE=0.9946, R²=0.0456) un polinomiālā (MAE=0.4888, RMSE=0.9993, R²=0.0364). Pārsteidzoši, bet **lineārā regresija uzrāda labāku R²** un zemāku RMSE, lai gan polinomiālajai ir nedaudz labāks MAE. Tas nozīmē, ka polinomiālais modelis nedod prognozēšanas priekšrocību šajā gadījumā.

**Vai polinomiālais modelis nopietni uzlaboja R²?**

Nē — polinomiālā regresija pat samazināja R² (no 0.0456 uz 0.0364) un pasliktināja RMSE. Tas notiek, jo degree=2 ar 26 features ģenerē ~700+ polinomiālos features — modelis ir pārāk sarežģīts šim datu apjomam un sāk pārmācīties uz treniņa datiem, nevis uzlabot ģeneralizāciju.

**Vai ir overfitting pazīmes?**

Lineārajai regresijai: Train R²=0.0568, Test R²=0.0456, starpība=0.0112 — pieņemama. Polinomiālajai: Train R²=0.1334, Test R²=0.0364, starpība=0.0971 — tuvu robežai, bet formāli pieņemama (< 0.10). Tomēr polinomiālā modeļa Train R² ir gandrīz 4× augstāks nekā Test R², kas ir spēcīga overfitting pazīme — modelis iemācījās treniņa datus, bet nespēj ģeneralizēt.

**Ko ieteiktu biznesa vidē?**

Biznesa vidē ieteiktu sākt ar **lineāro regresiju** kā bāzlīniju — tā ir interpretējama un neuzrāda overfitting. Tomēr abiem modeļiem R² ir ļoti zems (~0.04–0.05), kas norāda, ka PageValues prognozēšanai lineāri modeļi nav piemēroti. Nākamais solis būtu izmēģināt **Random Forest** vai **Gradient Boosting** (piemēram, XGBoost), kas daudz labāk tiek galā ar asimetriskiem sadalījumiem un nelineārām sakarībām, kā arī Ridge/Lasso regularizāciju, lai mazinātu overfitting risku polinomiālajai regresijai.